In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

In [4]:
# 1. Cargar el dataset con tus 519 representantes élite
df_elite = pd.read_csv('../results/CMNPD_Representantes_Elite.csv')

moleculas_3d = []
compuestos_fallidos = 0

print(f"Iniciando conversión a 3D para {len(df_elite)} compuestos (esto tomará unos minutos)...")

# 2. Iterar sobre cada SMILES para generar su estructura 3D
for index, row in df_elite.iterrows():
    smiles = row['Smiles']
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is not None:
        try:
            # Paso A: Añadir hidrógenos
            mol_h = Chem.AddHs(mol)
            
            # Paso B: Generar coordenadas 3D iniciales con ETKDG
            AllChem.EmbedMolecule(mol_h, AllChem.ETKDG())
            
            # Paso C: Minimización de energía con el campo de fuerza MMFF94 (max 200 iteraciones)
            AllChem.MMFFOptimizeMolecule(mol_h, maxIters=200)
            
            # Etiquetar la molécula con su ID y Predicción para no perder el rastro
            mol_h.SetProp("_Name", f"CMNPD_Elite_{index}")
            mol_h.SetProp("pChEMBL_Score", str(row['Media_Predicciones']))
            
            moleculas_3d.append(mol_h)
        except Exception as e:
            compuestos_fallidos += 1
    else:
        compuestos_fallidos += 1

print(f"\n¡Proceso finalizado!")
print(f"Moléculas convertidas con éxito: {len(moleculas_3d)}")
if compuestos_fallidos > 0:
    print(f"Moléculas descartadas por fallos estéricos o de valencia: {compuestos_fallidos}")

Iniciando conversión a 3D para 519 compuestos (esto tomará unos minutos)...


[11:51:06] UFFTYPER: Unrecognized charge state for atom: 1
[11:51:06] UFFTYPER: Unrecognized charge state for atom: 26
[11:51:11] UFFTYPER: Unrecognized charge state for atom: 58
[11:51:16] UFFTYPER: Unrecognized charge state for atom: 35
[11:51:26] UFFTYPER: Unrecognized charge state for atom: 15
[11:51:56] UFFTYPER: Unrecognized charge state for atom: 0
[11:51:56] UFFTYPER: Unrecognized atom type: Zn+2 (0)
[11:54:15] UFFTYPER: Unrecognized charge state for atom: 1
[11:54:17] UFFTYPER: Unrecognized charge state for atom: 25
[11:54:32] UFFTYPER: Unrecognized charge state for atom: 20
[11:54:32] UFFTYPER: Unrecognized charge state for atom: 20
[11:54:37] UFFTYPER: Unrecognized charge state for atom: 1
[11:55:20] UFFTYPER: Unrecognized charge state for atom: 1
[11:55:45] UFFTYPER: Unrecognized charge state for atom: 1
[11:55:47] UFFTYPER: Unrecognized charge state for atom: 6



¡Proceso finalizado!
Moléculas convertidas con éxito: 515
Moléculas descartadas por fallos estéricos o de valencia: 4


In [5]:
# 3. Guardar todas las estructuras 3D en un único archivo SDF
archivo_salida = '../results/CMNPD_Elite_3D.sdf'
writer = Chem.SDWriter(archivo_salida)
for mol in moleculas_3d:
    writer.write(mol)
writer.close()